# Enhanced Skills Taxonomy Drift Analysis

This notebook provides a more robust analysis to address the limitations identified in the initial drift analysis.

## Issues with Initial Analysis
1. **All skills showing version 9.31**: Suggests either a mass refresh or data quality issues
2. **No meaningful drift detected**: 100% of skills appear current
3. **Limited temporal analysis**: Not leveraging processed_at timestamps
4. **Missing skills**: 32 skills in use but not in comprehensive library

## Enhanced Analysis Approach
1. **Temporal Analysis**: Use `processed_at` dates to understand skill age
2. **Version Pattern Analysis**: Deep dive into version distribution patterns  
3. **Missing Skills Investigation**: Analyse the 32 missing skills
4. **Alternative Drift Metrics**: Consider skill age vs. version numbers
5. **Historical Context**: Understand Lightcast's update patterns


In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set up plotting parameters
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Load data from previous analysis
print("Loading data...")
skills_comprehensive = pd.read_csv('../data/skills_library/lightcast_skills_comprehensive.csv')
job_skill_mapping = pd.read_csv('../data/input_data/job_skill_mapping.csv')

print(f"Data loaded: {len(skills_comprehensive):,} skills, {len(job_skill_mapping):,} job-skill mappings")

# Get skills in use
skills_in_use = set(job_skill_mapping['Skill_ID'].unique())
print(f"Unique skills in use: {len(skills_in_use):,}")

# Basic preprocessing
skills_comprehensive['latest_version_numeric'] = pd.to_numeric(skills_comprehensive['latest_version'], errors='coerce')
skills_comprehensive['is_in_use'] = skills_comprehensive['skill_id'].isin(skills_in_use)


Loading data...
Data loaded: 38,395 skills, 40,170 job-skill mappings
Unique skills in use: 2,091


## 1. Deep Dive into Version Distribution Patterns


In [3]:
# Detailed version analysis
print("=== COMPREHENSIVE VERSION ANALYSIS ===")

# Overall version distribution
version_dist = skills_comprehensive['latest_version_numeric'].value_counts().sort_index()
print(f"\nComplete version distribution (all {len(skills_comprehensive):,} skills):")
for version, count in version_dist.items():
    percentage = (count / len(skills_comprehensive)) * 100
    print(f"Version {version}: {count:,} skills ({percentage:.1f}%)")

# Version distribution for skills in use
skills_in_use_subset = skills_comprehensive[skills_comprehensive['is_in_use']]
version_dist_in_use = skills_in_use_subset['latest_version_numeric'].value_counts().sort_index()
print(f"\nVersion distribution for skills in use ({len(skills_in_use_subset):,} skills):")
for version, count in version_dist_in_use.items():
    percentage = (count / len(skills_in_use_subset)) * 100
    print(f"Version {version}: {count:,} skills ({percentage:.1f}%)")

# Key insight: Is version 9.31 a mass update?
v931_total = version_dist.get(9.31, 0)
v931_in_use = version_dist_in_use.get(9.31, 0)
print(f"\n🔍 KEY INSIGHT:")
print(f"Version 9.31 represents {(v931_total/len(skills_comprehensive)*100):.1f}% of ALL skills")
print(f"Version 9.31 represents {(v931_in_use/len(skills_in_use_subset)*100):.1f}% of skills IN USE")
print(f"This suggests a potential mass refresh or data filtering issue")

=== COMPREHENSIVE VERSION ANALYSIS ===

Complete version distribution (all 38,395 skills):
Version 8.0: 10 skills (0.0%)
Version 8.1: 49 skills (0.1%)
Version 8.11: 41 skills (0.1%)
Version 8.12: 12 skills (0.0%)
Version 8.13: 15 skills (0.0%)
Version 8.14: 2,025 skills (5.3%)
Version 8.15: 4 skills (0.0%)
Version 8.16: 857 skills (2.2%)
Version 8.17: 3 skills (0.0%)
Version 8.18: 14 skills (0.0%)
Version 8.19: 4 skills (0.0%)
Version 8.2: 108 skills (0.3%)
Version 8.21: 6 skills (0.0%)
Version 8.22: 5 skills (0.0%)
Version 8.23: 5 skills (0.0%)
Version 8.24: 3 skills (0.0%)
Version 8.25: 8 skills (0.0%)
Version 8.26: 4 skills (0.0%)
Version 8.27: 10 skills (0.0%)
Version 8.28: 4 skills (0.0%)
Version 8.29: 2 skills (0.0%)
Version 8.3: 44 skills (0.1%)
Version 8.32: 3 skills (0.0%)
Version 8.34: 2 skills (0.0%)
Version 8.36: 1 skills (0.0%)
Version 8.4: 112 skills (0.3%)
Version 8.5: 81 skills (0.2%)
Version 8.6: 61 skills (0.2%)
Version 8.7: 49 skills (0.1%)
Version 8.8: 67 skills (0.

## 2. Temporal Analysis Using processed_at Field


In [4]:
# Temporal analysis using processed_at field
print("=== TEMPORAL DRIFT ANALYSIS ===")

# Parse processed_at dates
skills_comprehensive['processed_at_clean'] = pd.to_datetime(skills_comprehensive['processed_at'], errors='coerce')
skills_with_dates = skills_comprehensive.dropna(subset=['processed_at_clean'])

print(f"Skills with processed_at timestamps: {len(skills_with_dates):,}")
print(f"Date range: {skills_with_dates['processed_at_clean'].min()} to {skills_with_dates['processed_at_clean'].max()}")

# Calculate skill age (days since processing)
current_date = datetime.now()
skills_with_dates['skill_age_days'] = (current_date - skills_with_dates['processed_at_clean']).dt.days
skills_with_dates['skill_age_months'] = skills_with_dates['skill_age_days'] / 30.44  # Average days per month

# Focus on skills in use with dates
skills_in_use_with_dates = skills_with_dates[skills_with_dates['is_in_use']]
print(f"Skills in use with date information: {len(skills_in_use_with_dates):,}")

if len(skills_in_use_with_dates) > 0:
    print(f"\nAge statistics for skills in use:")
    age_stats = skills_in_use_with_dates['skill_age_months'].describe()
    print(age_stats)
    
    # Categorise by age instead of version
    def categorise_by_age(age_months):
        if pd.isna(age_months):
            return 'Unknown Age'
        elif age_months <= 3:
            return 'Very Recent (≤3 months)'
        elif age_months <= 6:
            return 'Recent (3-6 months)'
        elif age_months <= 12:
            return 'Moderate Age (6-12 months)'
        elif age_months <= 24:
            return 'Old (1-2 years)'
        else:
            return 'Very Old (>2 years)'
    
    skills_in_use_with_dates['age_category'] = skills_in_use_with_dates['skill_age_months'].apply(categorise_by_age)
    
    age_distribution = skills_in_use_with_dates['age_category'].value_counts()
    print(f"\nAge-based drift analysis:")
    for category, count in age_distribution.items():
        percentage = (count / len(skills_in_use_with_dates)) * 100
        print(f"{category}: {count:,} skills ({percentage:.1f}%)")
else:
    print("No skills in use have processed_at timestamps - using alternative approach")


=== TEMPORAL DRIFT ANALYSIS ===
Skills with processed_at timestamps: 34,665
Date range: 2025-06-10 20:21:29.068558 to 2025-06-10 20:21:30.127124
Skills in use with date information: 2,059

Age statistics for skills in use:
count    2059.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: skill_age_months, dtype: float64

Age-based drift analysis:
Very Recent (≤3 months): 2,059 skills (100.0%)


## 3. Version Distribution Pattern Analysis


In [5]:
# Deep dive into version patterns within skills in use vs not in use
print("=== VERSION PATTERN ANALYSIS ===")

# Split skills into in-use and not-in-use
skills_in_use_df = skills_comprehensive[skills_comprehensive['is_in_use']].copy()
skills_not_in_use_df = skills_comprehensive[~skills_comprehensive['is_in_use']].copy()

print(f"Skills in use: {len(skills_in_use_df):,}")
print(f"Skills not in use: {len(skills_not_in_use_df):,}")

# Analyse version spread for each group
print(f"\n=== VERSION SPREAD COMPARISON ===")

def analyze_version_spread(df, name):
    version_stats = df['latest_version_numeric'].describe()
    print(f"\n{name} - Version Statistics:")
    print(f"  Mean: {version_stats['mean']:.3f}")
    print(f"  Std Dev: {version_stats['std']:.3f}")
    print(f"  Min: {version_stats['min']:.2f}")
    print(f"  Max: {version_stats['max']:.2f}")
    
    # Count versions
    version_counts = df['latest_version_numeric'].value_counts().sort_index()
    print(f"  Unique versions: {len(version_counts)}")
    
    # Show top 5 most common versions
    top_versions = df['latest_version_numeric'].value_counts().head()
    print(f"  Top 5 versions:")
    for version, count in top_versions.items():
        pct = (count / len(df)) * 100
        print(f"    v{version}: {count:,} ({pct:.1f}%)")
    
    return version_counts

print("📊 ANALYSIS RESULTS:")
in_use_versions = analyze_version_spread(skills_in_use_df, "SKILLS IN USE")
not_in_use_versions = analyze_version_spread(skills_not_in_use_df, "SKILLS NOT IN USE")

# Key insights about version bias
print(f"\n🔍 KEY INSIGHTS:")

# Check if we're biased toward newer versions
newer_versions = [9.31, 9.27, 9.26, 9.25]  # Recent versions
older_versions = [8.14, 8.16, 8.9, 8.8, 8.7]  # Older versions

newer_in_use = skills_in_use_df[skills_in_use_df['latest_version_numeric'].isin(newer_versions)]
older_in_use = skills_in_use_df[skills_in_use_df['latest_version_numeric'].isin(older_versions)]

newer_not_in_use = skills_not_in_use_df[skills_not_in_use_df['latest_version_numeric'].isin(newer_versions)]
older_not_in_use = skills_not_in_use_df[skills_not_in_use_df['latest_version_numeric'].isin(older_versions)]

print(f"• Skills in use from newer versions (9.25+): {len(newer_in_use):,} ({len(newer_in_use)/len(skills_in_use_df)*100:.1f}%)")
print(f"• Skills in use from older versions (8.x): {len(older_in_use):,} ({len(older_in_use)/len(skills_in_use_df)*100:.1f}%)")
print(f"• Skills NOT in use from newer versions (9.25+): {len(newer_not_in_use):,} ({len(newer_not_in_use)/len(skills_not_in_use_df)*100:.1f}%)")
print(f"• Skills NOT in use from older versions (8.x): {len(older_not_in_use):,} ({len(older_not_in_use)/len(skills_not_in_use_df)*100:.1f}%)")

# Calculate selection ratio
if len(skills_comprehensive[skills_comprehensive['latest_version_numeric'] == 9.31]) > 0:
    v931_total = len(skills_comprehensive[skills_comprehensive['latest_version_numeric'] == 9.31])
    v931_in_use = len(skills_in_use_df[skills_in_use_df['latest_version_numeric'] == 9.31])
    v931_usage_rate = (v931_in_use / v931_total) * 100
    print(f"• Usage rate for v9.31 skills: {v931_usage_rate:.1f}% ({v931_in_use:,} of {v931_total:,} v9.31 skills are in use)")

# Check if older versions have different usage patterns
older_version_skills = skills_comprehensive[skills_comprehensive['latest_version_numeric'] < 9.0]
if len(older_version_skills) > 0:
    older_in_use_count = len(older_version_skills[older_version_skills['is_in_use']])
    older_usage_rate = (older_in_use_count / len(older_version_skills)) * 100
    print(f"• Usage rate for pre-v9.0 skills: {older_usage_rate:.1f}% ({older_in_use_count:,} of {len(older_version_skills):,} older skills are in use)")

print(f"\n💡 INTERPRETATION:")
if len(older_in_use) == 0:
    print("• Your organisation is exclusively using the most recent skill versions (9.31)")
    print("• This could indicate either excellent taxonomy maintenance OR a data filtering issue")
    print("• Consider investigating: Are older skills intentionally excluded or is this a data artifact?")
else:
    print("• Your organisation uses a mix of skill versions")
    print("• This provides a more realistic drift analysis opportunity")


=== VERSION PATTERN ANALYSIS ===
Skills in use: 2,059
Skills not in use: 36,336

=== VERSION SPREAD COMPARISON ===
📊 ANALYSIS RESULTS:

SKILLS IN USE - Version Statistics:
  Mean: 9.310
  Std Dev: 0.000
  Min: 9.31
  Max: 9.31
  Unique versions: 1
  Top 5 versions:
    v9.31: 2,059 (100.0%)

SKILLS NOT IN USE - Version Statistics:
  Mean: 9.198
  Std Dev: 0.337
  Min: 8.00
  Max: 9.31
  Unique versions: 48
  Top 5 versions:
    v9.31: 32,561 (89.6%)
    v8.14: 2,025 (5.6%)
    v8.16: 857 (2.4%)
    v8.4: 112 (0.3%)
    v8.2: 108 (0.3%)

🔍 KEY INSIGHTS:
• Skills in use from newer versions (9.25+): 2,059 (100.0%)
• Skills in use from older versions (8.x): 0 (0.0%)
• Skills NOT in use from newer versions (9.25+): 32,587 (89.7%)
• Skills NOT in use from older versions (8.x): 3,100 (8.5%)
• Usage rate for v9.31 skills: 5.9% (2,059 of 34,620 v9.31 skills are in use)
• Usage rate for pre-v9.0 skills: 0.0% (0 of 3,711 older skills are in use)

💡 INTERPRETATION:
• Your organisation is exclusive

## 4. Role-Based Skills Version Analysis


In [6]:
# Role-based analysis of skills versions
print("=== ROLE-BASED SKILLS VERSION ANALYSIS ===")

# Create comprehensive mapping of jobs to skills with version info
job_skill_enhanced = job_skill_mapping.merge(
    skills_comprehensive[['skill_id', 'name', 'latest_version_numeric', 'category_name', 'type_name']], 
    left_on='Skill_ID', 
    right_on='skill_id', 
    how='left'
)

print(f"Job-skill mappings with version info: {len(job_skill_enhanced):,}")
print(f"Successful skill matches: {job_skill_enhanced['latest_version_numeric'].notna().sum():,}")
print(f"Unmatched skills: {job_skill_enhanced['latest_version_numeric'].isna().sum():,}")

# Remove unmatched skills for analysis
job_skill_clean = job_skill_enhanced.dropna(subset=['latest_version_numeric']).copy()

# Role-level aggregations
print(f"\n=== ROLE-LEVEL VERSION ANALYSIS ===")

role_stats = job_skill_clean.groupby('JobProfileID').agg({
    'latest_version_numeric': ['count', 'mean', 'min', 'max', 'std'],
    'Skill_ID': 'count'
}).round(3)

# Flatten column names
role_stats.columns = ['skill_count', 'avg_version', 'min_version', 'max_version', 'version_std', 'total_skills']
role_stats = role_stats.reset_index()

print(f"Analyzed {len(role_stats):,} job profiles")
print(f"Average skills per role: {role_stats['skill_count'].mean():.1f}")

# Role version statistics
print(f"\nRole-level version statistics:")
print(f"Average version across all roles: {role_stats['avg_version'].mean():.3f}")
print(f"Standard deviation of role averages: {role_stats['avg_version'].std():.3f}")

# Identify roles with version diversity (mixed old and new skills)
diverse_roles = role_stats[role_stats['version_std'] > 0.1]  # Roles with some version diversity
homogeneous_roles = role_stats[role_stats['version_std'] <= 0.1]  # Roles with uniform versions

print(f"\nRole version diversity:")
print(f"• Roles with diverse skill versions: {len(diverse_roles):,} ({len(diverse_roles)/len(role_stats)*100:.1f}%)")
print(f"• Roles with homogeneous skill versions: {len(homogeneous_roles):,} ({len(homogeneous_roles)/len(role_stats)*100:.1f}%)")

# Find most and least up-to-date roles
top_current_roles = role_stats.nlargest(10, 'avg_version')[['JobProfileID', 'avg_version', 'skill_count']]
least_current_roles = role_stats.nsmallest(10, 'avg_version')[['JobProfileID', 'avg_version', 'skill_count']]

print(f"\n📈 TOP 10 MOST UP-TO-DATE ROLES (highest avg version):")
for idx, role in top_current_roles.iterrows():
    print(f"• {role['JobProfileID']}: v{role['avg_version']:.3f} avg ({int(role['skill_count'])} skills)")

print(f"\n📉 TOP 10 LEAST UP-TO-DATE ROLES (lowest avg version):")
for idx, role in least_current_roles.iterrows():
    print(f"• {role['JobProfileID']}: v{role['avg_version']:.3f} avg ({int(role['skill_count'])} skills)")

# Categorize roles by their version profile
def categorize_role_currency(avg_version, version_std):
    if avg_version >= 9.3:
        return "Highly Current (v9.3+)"
    elif avg_version >= 9.0:
        return "Current (v9.0-9.3)"
    elif avg_version >= 8.5:
        return "Moderately Current (v8.5-9.0)"
    else:
        return "Outdated (v<8.5)"

role_stats['currency_category'] = role_stats.apply(
    lambda x: categorize_role_currency(x['avg_version'], x['version_std']), axis=1
)

currency_distribution = role_stats['currency_category'].value_counts()
print(f"\n🎯 ROLE CURRENCY DISTRIBUTION:")
for category, count in currency_distribution.items():
    percentage = (count / len(role_stats)) * 100
    print(f"• {category}: {count:,} roles ({percentage:.1f}%)")

# Skills per category analysis
category_impact = job_skill_clean.groupby('category_name').agg({
    'latest_version_numeric': ['count', 'mean', 'std'],
    'JobProfileID': 'nunique'
}).round(3)

category_impact.columns = ['skill_instances', 'avg_version', 'version_std', 'roles_using']
category_impact = category_impact.reset_index().sort_values('avg_version', ascending=True)

print(f"\n📊 SKILLS CATEGORY VERSION ANALYSIS (Bottom 10):")
print("Categories with lowest average versions (potential update opportunities):")
bottom_categories = category_impact.head(10)
for idx, cat in bottom_categories.iterrows():
    if pd.notna(cat['category_name']):
        print(f"• {cat['category_name']}: v{cat['avg_version']:.3f} avg ({int(cat['skill_instances'])} instances, {int(cat['roles_using'])} roles)")

print(f"\n💡 KEY RECOMMENDATIONS:")
outdated_roles = role_stats[role_stats['avg_version'] < 9.0]
if len(outdated_roles) > 0:
    print(f"• {len(outdated_roles):,} roles have below v9.0 average - prioritise for review")
    print(f"• Focus on roles: {', '.join(outdated_roles.head(5)['JobProfileID'].tolist())}")
else:
    print("• All roles appear to use current skill versions (v9.0+)")
    print("• Consider investigating if this represents actual currency or data filtering")

if len(diverse_roles) > 0:
    print(f"• {len(diverse_roles):,} roles show version diversity - good candidates for targeted updates")
else:
    print("• No version diversity detected - may indicate mass update or filtering issue")


=== ROLE-BASED SKILLS VERSION ANALYSIS ===
Job-skill mappings with version info: 40,170
Successful skill matches: 40,045
Unmatched skills: 125

=== ROLE-LEVEL VERSION ANALYSIS ===
Analyzed 715 job profiles
Average skills per role: 56.0

Role-level version statistics:
Average version across all roles: 9.310
Standard deviation of role averages: 0.000

Role version diversity:
• Roles with diverse skill versions: 0 (0.0%)
• Roles with homogeneous skill versions: 715 (100.0%)

📈 TOP 10 MOST UP-TO-DATE ROLES (highest avg version):
• R0001.5: v9.310 avg (68 skills)
• R0001.6: v9.310 avg (68 skills)
• R0002.0: v9.310 avg (49 skills)
• R0002.1: v9.310 avg (49 skills)
• R0002.2: v9.310 avg (49 skills)
• R0002.3: v9.310 avg (49 skills)
• R0002.4: v9.310 avg (49 skills)
• R0003.0: v9.310 avg (34 skills)
• R0003.2: v9.310 avg (34 skills)
• R0005.1: v9.310 avg (39 skills)

📉 TOP 10 LEAST UP-TO-DATE ROLES (lowest avg version):
• R0001.5: v9.310 avg (68 skills)
• R0001.6: v9.310 avg (68 skills)
• R000

In [ ]:
# Visualizations for role-based analysis
print("=== ROLE-BASED ANALYSIS VISUALIZATIONS ===")

# Create visualizations
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# 1. Distribution of average versions across roles
ax1.hist(role_stats['avg_version'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
ax1.set_xlabel('Average Skill Version per Role')
ax1.set_ylabel('Number of Roles')
ax1.set_title('Distribution of Average Skill Versions Across Job Roles')
ax1.axvline(role_stats['avg_version'].mean(), color='red', linestyle='--', label=f'Mean: {role_stats["avg_version"].mean():.3f}')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Role currency categories pie chart
currency_counts = role_stats['currency_category'].value_counts()
colors = ['green', 'lightgreen', 'orange', 'red'][:len(currency_counts)]
ax2.pie(currency_counts.values, labels=currency_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
ax2.set_title('Distribution of Role Currency Categories')

# 3. Skills per role vs average version scatter
ax3.scatter(role_stats['skill_count'], role_stats['avg_version'], alpha=0.6, s=30)
ax3.set_xlabel('Number of Skills per Role')
ax3.set_ylabel('Average Skill Version')
ax3.set_title('Role Complexity vs Skill Currency')
ax3.grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(role_stats['skill_count'], role_stats['avg_version'], 1)
p = np.poly1d(z)
ax3.plot(role_stats['skill_count'], p(role_stats['skill_count']), "r--", alpha=0.8)

# 4. Version diversity (standard deviation) across roles
ax4.hist(role_stats['version_std'], bins=20, alpha=0.7, color='lightcoral', edgecolor='black')
ax4.set_xlabel('Version Standard Deviation per Role')
ax4.set_ylabel('Number of Roles')
ax4.set_title('Version Diversity Distribution Across Roles')
ax4.axvline(role_stats['version_std'].mean(), color='red', linestyle='--', label=f'Mean: {role_stats["version_std"].mean():.3f}')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Additional analysis: Role clustering by skills profile
print(f"\n=== ROLE CLUSTERING INSIGHTS ===")

# Identify role archetypes based on skills patterns
high_skill_roles = role_stats[role_stats['skill_count'] > role_stats['skill_count'].quantile(0.75)]
low_skill_roles = role_stats[role_stats['skill_count'] < role_stats['skill_count'].quantile(0.25)]

print(f"High-complexity roles (top 25% by skill count): {len(high_skill_roles):,} roles")
print(f"• Average skills: {high_skill_roles['skill_count'].mean():.1f}")
print(f"• Average version: {high_skill_roles['avg_version'].mean():.3f}")
print(f"• Sample roles: {', '.join(high_skill_roles.head(5)['JobProfileID'].tolist())}")

print(f"\nLow-complexity roles (bottom 25% by skill count): {len(low_skill_roles):,} roles")
print(f"• Average skills: {low_skill_roles['skill_count'].mean():.1f}")
print(f"• Average version: {low_skill_roles['avg_version'].mean():.3f}")
print(f"• Sample roles: {', '.join(low_skill_roles.head(5)['JobProfileID'].tolist())}")

# Export role analysis for further action
if 'role_stats' in locals():
    output_dir = '../data/output'
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    role_export = role_stats.sort_values('avg_version', ascending=True)
    role_export.to_csv(f'{output_dir}/role_skills_version_analysis.csv', index=False)
    print(f"\n✅ Role analysis exported to: {output_dir}/role_skills_version_analysis.csv")
    
    # Also export detailed job-skill mappings with versions
    job_skill_clean.to_csv(f'{output_dir}/job_skill_mapping_with_versions.csv', index=False)
    print(f"✅ Detailed job-skill mapping exported to: {output_dir}/job_skill_mapping_with_versions.csv")
